# llm_agent

> Language-model team members for Phase 10: `LLMMember` (one API call per event: cached rules, rendered history per memory setting, current observation; parse, apply, post the report, log the turn), belief elicitation, the scripted liars, and `runTurns`, the Phase 5 event queue with a turn budget per member.

In [ ]:
#| default_exp llm_agent

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import heapq
import math

import numpy as np

from explRL.adversary import Liar
from explRL.board_agents import BoardBandit, ClaimBoard
from explRL.harness import MessageBus
from explRL.multi import Member, Outcome, Record, TeamHistory

FRAMING = {'own': 'You are scored on the {u} you collect yourself.',
           'team': "You are scored on the team's total {u}, the sum over all teammates.",
           'credit': 'You are scored on the {u} you collect yourself plus a share of the {u} others collect at '
                     'locations you reported first.'}
BELIEF_SCHEMA = {'type': 'object', 'additionalProperties': False, 'required': ['estimates'],
                 'properties': {'estimates': {'type': 'array', 'items': {
                     'type': 'object', 'additionalProperties': False, 'required': ['location', 'value'],
                     'properties': {'location': {'type': 'string'}, 'value': {'type': 'number'}}}}}}
MEMORY = ('full', 'last8', 'notes')


def _plain(x):
    if isinstance(x, tuple):
        return [_plain(v) for v in x]
    return x.item() if isinstance(x, np.generic) else x


class LLMMember(Member):
    """A Phase 5 member whose every event is one API call: cached rules, then the rendered history and the current
    observation (decision L4); the reply is parsed, applied, and its report posted to the channel."""
    extraNames = ('valid', 'cached', 'cost', 'explore', 'reported')

    def __init__(self, client, env, channel=None, memory='full', framing='own', lessons='', home=None, strict=False):
        if memory not in MEMORY:
            raise ValueError(f'unknown memory {memory}')
        self.client, self.env, self.channel, self.memory = client, env, channel, memory
        self.framing, self.lessons, self.timeUnit, self.home, self.strict = framing, lessons, env.timeUnit, home, strict
        self.past, self.notes, self.log, self._system = [], '', [], None

    def system(self):
        if self._system is None:
            s = (self.env.rules(self.id) if getattr(self.env, 'slotRules', False) else self.env.rules()) + '\n\n' + FRAMING[self.framing].format(u=self.env.unit)
            if self.lessons:
                s += '\n\nLessons you wrote at the end of earlier episodes (the world changes between episodes):\n' + \
                     self.lessons
            if self.home:
                s += '\n\nYour squad\'s home range is the locations ' + ', '.join(self.env.names[j] for j in sorted(self.home)) + \
                     ' (an arc of the ring); ' + ('stay and work only there; read teammates\' reports from everywhere.'
                                                  if self.strict else 'prefer working there, but you may go anywhere.')
            self._system = s
        return self._system

    def history(self):
        "Past turns as text blocks per the memory setting; the cache breakpoint on the last block."
        if self.memory == 'notes':
            texts = [f'Your notes so far: {self.notes}'] if self.notes else []
        else:
            texts = self.past if self.memory == 'full' else self.past[-8:]
            texts = ['Your past turns, oldest first:\n' + texts[0]] + texts[1:] if texts else []
        blocks = [{'type': 'text', 'text': x} for x in texts]
        if blocks:
            blocks[-1]['cache_control'] = {'type': 'ephemeral'}
        return blocks

    def prepare(self, world, t):
        "Observe at `t` and render the request (serial: reads and updates the channel's per-agent state)."
        obs = self.env.observe(self.id, t, self.channel)
        return obs, self.system(), self.history() + [{'type': 'text', 'text': 'Now:\n' + obs}]

    def decide(self, prep):
        "The API call (may run concurrently with other members' calls in a round)."
        obs, system, blocks = prep
        return obs, self.client.complete(system, blocks, self.env.SCHEMA, task=self.env.task)

    def commit(self, world, t, dec):
        "Parse, apply at `t`, post the report and log the turn."
        obs, reply = dec
        action, report, valid = self.env.parse(reply.parsed, self.id)
        reward, dt, loc, info = world.apply(self.id, action, t)
        self.env.post(self.id, report, t, self.channel, info)
        explore = bool(self.env.explored(self.id, action, info))
        if isinstance(reply.parsed, dict) and isinstance(reply.parsed.get('notes'), str):
            self.notes = reply.parsed['notes'][:1000]
        self.past.append(self.env.summary(self.id, t, action, reward, report, info))
        self.log.append(dict(kind='turn', agent=self.id, t=_plain(t), obs=obs, response=reply.text,
                             action=_plain(action), valid=valid, reward=_plain(reward), report=_plain(report),
                             explore=explore, stop=reply.stopReason, refusal=reply.refusal, usage=reply.usage,
                             cost=reply.cost))
        return Outcome(dt, info['kind'], loc, reward, (valid, reply.cached, reply.cost, explore, report is not None))

    def event(self, world, t):
        return self.commit(world, t, self.decide(self.prepare(world, t)))


def elicitBeliefs(member):
    "Text foraging: one request for the agent's estimate of each location's value -> Phase 8 {'mean', 'n'}."
    env = member.env
    q = ('The run is over. For each location you have an opinion about, estimate the reward per second a lone '
         'forager would get by staying there now (its food rate, or a negative value for danger). Leave out the '
         'locations you know nothing about. Reply with JSON: estimates is a list of {location, value}.')
    reply = member.client.complete(member.system(), member.history() + [{'type': 'text', 'text': q}], BELIEF_SCHEMA,
                                   task=env.task)
    mean, n = np.zeros(env.world.L), np.zeros(env.world.L)
    for e in (reply.parsed or {}).get('estimates', []) if isinstance(reply.parsed, dict) else []:
        j, v = env.index.get(str(e.get('location', '')).strip().lower()), e.get('value')
        if j is not None and isinstance(v, (int, float)) and math.isfinite(v):
            mean[j], n[j] = float(v), 1.0
    member.log.append(dict(kind='beliefs', agent=member.id, response=reply.text, usage=reply.usage, cost=reply.cost,
                           stop=reply.stopReason, refusal=reply.refusal))
    return dict(mean=mean.tolist(), n=n.tolist())


class BoardLiar(BoardBandit):
    """Scripted task-board liar (the Phase 9 rule; no API calls): works honestly (it knows the answer and gets it right
    with probability `solveRate`), but claims 'solved' on an open task it has not solved and 'working' on one it
    does not open."""
    def __init__(self, channel, rng, solveRate=0.5):
        super().__init__(channel, rng)
        self.solveRate = float(solveRate)

    def choose(self, b, t):
        a = super().choose(b, t)
        if a[0] == 'submit':
            p = b.tasks[b.opened[self.id]].puzzle
            a = ('submit', p.answer if self.rng.random() < self.solveRate else -1)
        return a

    def post(self, b, action, info, t):
        others = [k for k in b.open if k != b.opened[self.id]]
        if not others:
            return
        k = int(others[int(self.rng.integers(len(others)))])
        kind = 'working' if info.get('result') == 'opened' else 'solved'
        if isinstance(self.channel, ClaimBoard):
            self.channel.post(self.id, k, kind, t)
        elif isinstance(self.channel, MessageBus) and self.channel.due(self.id, t):
            self.channel.send(b, self.id, k, 1.0 if kind == 'working' else 2.0, t)


def ScriptedLiarText(task, channel, rng):
    "The scripted liar for a text task: the Phase 9 `Liar` (foraging) or a `BoardLiar` (task board)."
    return Liar(channel) if task == 'forage' else BoardLiar(channel, rng)


def runTurns(members, world, turns, t0=0.0):
    "The Phase 5 event queue in which member i leaves after `turns[i]` events (an int applies to all)."
    turns = [int(turns)] * len(members) if np.isscalar(turns) else [int(x) for x in turns]
    for i, m in enumerate(members):
        m.id = i
    hist = TeamHistory([m.extraNames for m in members])
    heap, done = [(t0, i) for i in range(len(members)) if turns[i] > 0], [0] * len(members)
    heapq.heapify(heap)
    while heap:
        t, i = heapq.heappop(heap)
        out = members[i].event(world, t)
        if not out.dt > 0:
            raise ValueError(f'event duration must be > 0, got {out.dt}')
        hist.add(Record(i, t, out.dt, out.kind, out.location, out.reward, out.extras))
        done[i] += 1
        if done[i] < turns[i]:
            heapq.heappush(heap, (t + out.dt, i))
    return hist

## Tests

In [ ]:
import json, re
import numpy as np
from explRL.llm_agent import *
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient
from explRL.llm_env import TextForagingEnv, locationNames
from explRL.llm_board import TextBoardEnv
from explRL.board_agents import BoardLead, ClaimBoard
from explRL.harness import Blackboard, Lead, MessageBus
from explRL.harness_runs import benchWorld
from explRL.multi import spawnRngs
from explRL.taskboard import TaskBoardEnv

def obsOf(req):
    return req['messages'][0]['content'][-1]['text']

def forager(req):
    "Stay where you are and report it; every third turn go to the coordinator's suggestion or the first board entry."
    o = obsOf(req)
    here = re.search(r'You are at (\w+)', o).group(1)
    sug = re.search(r'suggests: (\w+)', o) or re.search(r'Board[^:]*: (\w+) ', o)
    go = sug and int(float(re.search(r'Time ([\d.]+)', o).group(1))) % 3 == 0 and sug.group(1) != 'empty'
    return json.dumps(dict(action='go' if go else 'stay', target=sug.group(1) if go else '', reportLocation=here,
                           reportValue=0.5, notes=f'at {here}'))

def team(channel, n=3, turns=6, memory='full', seed=0):
    w = benchWorld(); w.reset(seed, n)
    env = TextForagingEnv(w, locationNames(64, seed))
    fc = FakeClient(forager)
    ms = [LLMMember(LLMClient('claude-haiku-5-5', client=fc), env, channel, memory=memory) for _ in range(n)]
    hist = runTurns(ms, w, turns)
    return ms, hist, fc, w

# every agent makes exactly its turn budget; extras record validity and reports
ms, hist, fc, w = team(None, turns=5)
assert [len(hist.forAgent(i)) for i in range(3)] == [5, 5, 5] and len(fc.requests) == 15
assert all(r.extras[0] and r.extras[4] for r in hist.records)
w = benchWorld(); w.reset(0, 3); env = TextForagingEnv(w, locationNames(64, 0))
h = runTurns([LLMMember(LLMClient('claude-haiku-5-5', client=FakeClient(forager)), env) for _ in range(3)], w, [2, 4, 1])
assert [len(h.forAgent(i)) for i in range(3)] == [2, 4, 1]

# the same script gives identical logs
a, b = team(Blackboard(64, None, 0.01, 3))[0], team(Blackboard(64, None, 0.01, 3))[0]
assert json.dumps([m.log for m in a]) == json.dumps([m.log for m in b])

# channels: reports reach the blackboard, messages reach two teammates, the lead's suggestion reaches every agent
bb = Blackboard(64, None, 0.01, 3)
ms, *_ = team(bb)
assert sum(bb.stats[i]['written'] for i in range(3)) == 18
assert any('Board (' in r['obs'] and 'empty' not in r['obs'] for r in ms[1].log)
mb = MessageBus(3, 2, 10.0, np.inf, spawnRngs(5, 3))
ms, *_ = team(mb, turns=12)
assert sum(mb.stats[i]['sent'] for i in range(3)) > 0 and any('agent ' in r['obs'] for m in ms for r in m.log)
lead = Lead(64, 3)
ms, *_ = team(lead)
assert all('The coordinator suggests' in r['obs'] for m in ms for r in m.log) and lead.table.C.sum() > 0

# memory: full is append-only (each turn's history starts with the previous turn's), last8 keeps 8, notes keeps notes
ms, hist, fc, w = team(None, n=1, turns=12)
hs = [[c['text'] for c in r['messages'][0]['content'][:-1]] for r in fc.requests]
assert all(hs[k + 1][:len(hs[k])] == hs[k] and len(hs[k + 1]) == len(hs[k]) + 1 for k in range(11))
assert all('cache_control' in r['messages'][0]['content'][-2] for r in fc.requests[1:])
assert len({json.dumps(r['system']) for r in fc.requests}) == 1                        # the cached prefix never changes
ms, hist, fc, w = team(None, n=1, turns=12, memory='last8')
assert [len(r['messages'][0]['content']) - 1 for r in fc.requests] == [0, 1, 2, 3, 4, 5, 6, 7, 8, 8, 8, 8]
ms, hist, fc, w = team(None, n=1, turns=3, memory='notes')
assert [c['text'] for c in fc.requests[2]['messages'][0]['content'][:-1]] == [f'Your notes so far: {ms[0].log[1]["response"] and json.loads(ms[0].log[1]["response"])["notes"]}']

# invalid replies, refusals and max_tokens stops become 'stay' and are counted
w = benchWorld(); w.reset(0, 1); env = TextForagingEnv(w, locationNames(64, 0))
fc = FakeClient([dict(text='{"action": "go", "target": "Atlantis", "reportLocation": "", "reportValue": 0, "notes": ""}'),
                 dict(text='', stop='refusal', refusal='other'), dict(text='{"act', stop='max_tokens')])
m = LLMMember(LLMClient('claude-haiku-5-5', client=fc), env)
h = runTurns([m], w, 3)
assert [r.kind for r in h.records] == ['stay'] * 3 and [r.extras[0] for r in h.records] == [False] * 3
assert [x['refusal'] for x in m.log] == [None, 'other', None] and [x['stop'] for x in m.log][2] == 'max_tokens'

# framing and lessons change only the system prompt
w = benchWorld(); w.reset(0, 1); env = TextForagingEnv(w, locationNames(64, 0))
s = {f: LLMMember(None, env, framing=f).system() for f in FRAMING}
assert len(set(s.values())) == 3 and all(x.startswith(env.rules()) for x in s.values())
assert 'team\'s total food' in s['team'] and 'Lessons' in LLMMember(None, env, lessons='go east').system()
hs = LLMMember(None, env, home={3, 4}).system()
assert 'stay and work only there' in LLMMember(None, env, home={3, 4}, strict=True).system() and 'prefer working there' in hs
assert f"home range is the locations {env.names[3]}, {env.names[4]}" in hs and 'home range' not in s['own']

# beliefs: named estimates map to locations; unknown names and non-numbers are dropped
names = env.names
fc = FakeClient([json.dumps(dict(estimates=[dict(location=names[3], value=0.8), dict(location='Nowhere', value=1),
                                            dict(location=names[5], value=-2.6)]))])
m = LLMMember(LLMClient('claude-haiku-5-5', client=fc), env); m.id = 0
e = elicitBeliefs(m)
assert e['n'][3] == e['n'][5] == 1 and sum(e['n']) == 2 and e['mean'][3] == 0.8 and e['mean'][5] == -2.6
assert m.log[-1]['kind'] == 'beliefs' and fc.requests[0]['output_config']['format']['schema'] == BELIEF_SCHEMA

# the board: a fake-client solver opens, then submits the right answer (read from the observation's puzzle)
def solver(req):
    o = obsOf(req)
    m = re.search(r'Your open subtask: T(\d+), open', o)
    if m:
        f, rng, y = re.search(r'return (.*)\n', o).group(1), int(re.search(r'x < (\d+)', o).group(1)), \
            int(re.search(r'f\(x\) == (-?\d+)', o).group(1))
        x = next(x for x in range(rng) if eval(f, {'x': x}) == y)
        return json.dumps(dict(action='submit', target='', answer=x, claimTask=f'T{m.group(1)}', claimKind='solved', notes=''))
    first = re.search(r'Open subtasks[^:]*: T(\d+)', o).group(1)
    return json.dumps(dict(action='open', target=f'T{first}', answer=0, claimTask=f'T{first}', claimKind='working', notes=''))
b = TaskBoardEnv(0, 2, difficulty=1); cb = ClaimBoard()
env = TextBoardEnv(b)
ms = [LLMMember(LLMClient('claude-haiku-5-5', client=FakeClient(solver)), env, cb) for _ in range(2)]
h = runTurns(ms, b, 6)
assert b.counts['solved'] >= 3 and b.counts['wrong'] == 0 and all(r.extras[0] for r in h.records)
assert any('Claims: T' in r['obs'] for r in ms[1].log)

# scripted liars: the Phase 9 liar for foraging; the board liar posts claims on tasks it is not working on
assert isinstance(ScriptedLiarText('forage', None, None), Liar)
b = TaskBoardEnv(1, 2, difficulty=1); cb = ClaimBoard()
liar = ScriptedLiarText('board', cb, np.random.default_rng(0))
runTurns([liar], b, 6)
assert cb.stats[0]['written'] == 6 and all(k != b.opened[0] for k, *_ in cb.view(1, 6.0, b.open) if _[0] == 'working')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()